# Kiên Đoàn TTS — Minh Anh

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~5 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Nhấn ▶** bên cạnh ô phía dưới — nút khởi động lớn sẽ hiện ra
2. **Nhấn "Khởi động — Chạy tất cả"** — tự động cài đặt & khởi động
3. Đợi Gradio khởi động → nhập văn bản → Click **Tạo giọng nói**
4. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:#F0EDFF;border:2px solid #5B3DF6;border-radius:16px;padding:28px 24px;text-align:center;max-width:480px;margin:12px auto">
  <svg width="36" height="36" viewBox="0 0 24 24" fill="none"
    stroke="#5B3DF6" stroke-width="2" stroke-linecap="round"
    stroke-linejoin="round" style="margin-bottom:12px;display:block;margin-left:auto;margin-right:auto">
    <path d="M12 2a3 3 0 0 0-3 3v7a3 3 0 0 0 6 0V5a3 3 0 0 0-3-3z"/>
    <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
    <line x1="12" y1="19" x2="12" y2="22"/>
  </svg>
  <div style="font-weight:700;font-size:20px;color:#1a1a2e;margin-bottom:6px">
    Kiên Đoàn TTS
  </div>
  <div style="font-size:14px;color:#555;margin-bottom:22px;line-height:1.6">
    Nhấn nút bên dưới để tự động cài đặt &amp; khởi động
  </div>
  <button
    onclick="google.colab.notebook.runAll()"
    style="background:#5B3DF6;color:#fff;border:none;border-radius:12px;padding:18px 0;font-size:17px;font-weight:700;cursor:pointer;width:100%;display:block;letter-spacing:0.3px">
    &#9654; &nbsp; Khởi động — Chạy tất cả
  </button>
  <div style="font-size:12px;color:#999;margin-top:14px">
    Lần đầu ~5 phút &nbsp;·&nbsp; Lần sau ~30 giây
  </div>
</div>
"""))


In [ ]:
print('Đang cài đặt (~2 phút)...')
!pip install -q omnivoice gradio numpy torch
print('Cài đặt hoàn tất!')

# Tải giọng mẫu 10s từ voice-notebooks repo
!wget -q https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/minh-anh.mp3 -O voice_sample.mp3
print('Đã tải voice sample!')


In [ ]:
print('Đang khởi động Omnivoice... (lần đầu ~5 phút, lần sau ~30 giây)')

import logging, os, re, time
import numpy as np
import torch
import gradio as gr
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Wait for GPU
for i in range(30):
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        break
    time.sleep(1)
else:
    print('GPU not available — make sure Runtime > Change runtime type > T4 GPU')

# Load model
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
logger.info(f'Model ready — SR: {SAMPLING_RATE}Hz')

# Voice clone prompt
logger.info('Creating VoiceClonePrompt...')
VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio='voice_sample.mp3')
logger.info('Voice prompt ready — Minh Anh')

# Generate function
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)

def generate_voice(text: str):
    text = text.strip()
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        return None
    if len(paragraphs) == 1:
        audio = model.generate(
            text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
            language='vi', speed=0.95, generation_config=GEN_CFG
        )[0]
    else:
        audios = []
        for i, p in enumerate(paragraphs):
            a = model.generate(
                text=p, voice_clone_prompt=VOICE_PROMPT,
                language='vi', speed=0.95, generation_config=GEN_CFG
            )[0]
            audios.append(a)
            if i < len(paragraphs) - 1:
                audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
        audio = np.concatenate(audios)
    waveform = (audio * 32767).astype(np.int16)
    return (SAMPLING_RATE, waveform)

# Launch UI
print('Khởi động giao diện Kiên Đoàn TTS — Minh Anh...')
CSS = ".gradio-container{{max-width:720px!important;margin:0 auto!important;padding:16px!important}}footer{{display:none!important}}"
THEME = gr.themes.Soft(primary_hue='indigo')
with gr.Blocks(title='Kiên Đoàn TTS — Minh Anh', theme=THEME, css=CSS) as demo:
    gr.Markdown('# Kiên Đoàn TTS\\nMinh Anh · Giọng nữ miền Bắc, nhẹ nhàng, truyền cảm, phù hợp marketing, quảng cáo, sách nói.')
    t = gr.Textbox(label='Nhập văn bản', lines=4, placeholder='Nhập văn bản bạn muốn chuyển thành giọng nói...')
    btn = gr.Button('Tạo giọng nói', variant='primary')
    out = gr.Audio(label='Kết quả')
    btn.click(generate_voice, inputs=[t], outputs=[out], concurrency_limit=1)

demo.launch(server_name='0.0.0.0', server_port=7860, share=True, theme=THEME, css=CSS)
